Las queries se han realizado en MongdoDB Compass. Habría que usar db.collection.aggregate() para realizar las siguientes consultas (siendo db la base de datos y collection la colección correspondiente):

## **1) Monstruos en las 10 habitaciones con más comentarios bug**

En la colección rooms

```python
[
  {
    $match: {
      hints: {
        $ne: null
      }
    }
  },
  {
    $addFields: {
      n_hints_bugs: {
        $size: {
          $filter: {
            input: "$hints",
            as: "hint",
            cond: {
              $eq: ["$$hint.category", "bug"]
            }
          }
        }
      }
    }
  },
  {
    $sort: {
      n_hints_bugs: -1
    }
  },
  {
    $limit: 10
  },
  {
    $unwind: "$monsters"
  },
  {
    $group: {
      _id: "$monsters.id",
      name: {
        $first: "$monsters.name"
      }
    }
  }
]

## **2) Mazmorras con comentarios hint por encima de la media**

En la colección rooms


```python
[
  {
    $addFields: {
      n_hints: {
        $size: {
          $filter: {
            input: "$hints",
            as: "hint",
            cond: {
              $eq: ["$$hint.category", "hint"]
            }
          }
        }
      }
    }
  },
  {
    $group: {
      _id: "$dungeon_id",
      dungeon_name: {
        $first: "$dungeon_name"
      },
      total_hints_mazmorra: {
        $sum: "$n_hints"
      }
    }
  },
  {
    $group: {
      _id: null,
      media_global: {
        $avg: "$total_hints_mazmorra"
      },
      mazmorras: {
        $push: {
          nombre: "$dungeon_name",
          total: "$total_hints_mazmorra"
        }
      }
    }
  },
  {
    $unwind: "$mazmorras"
  },
  {
    $match: {
      $expr: {
        $gt: ["$mazmorras.total", "$media_global"]
      }
    }
  },
  {
    $project: {
      _id: 0,
      dungeon_name: "$mazmorras.nombre"
    }
  }
]

## **3) Por mazmorra: comentarios por tipo, oro total, nivel mediano y usuarios por país**

En la colección rooms

```python
[
  {
    $group: {
      _id: "$dungeon_id",
      dungeon_name: { $first: "$dungeon_name" }
    }
  },
  {
    $lookup: {
      from: "rooms",
      let: { d_id: "$_id" },
      pipeline: [
        { $match: { $expr: { $eq: ["$dungeon_id", "$$d_id"] } } },
        { $unwind: { path: "$hints", preserveNullAndEmptyArrays: false } },
        { $group: { _id: "$hints.category", total: { $sum: 1 } } },
        { $project: { _id: 0, tipo: "$_id", total: 1 } }
      ],
      as: "comentarios_por_tipo"
    }
  },
  {
    $lookup: {
      from: "rooms",
      let: { d_id: "$_id" },
      pipeline: [
        { $match: { $expr: { $eq: ["$dungeon_id", "$$d_id"] } } },
        { $unwind: { path: "$hints", preserveNullAndEmptyArrays: false } },
        { $group: { _id: { email: "$hints.publish_by.email", pais: "$hints.publish_by.country" } } },
        { $group: { _id: "$_id.pais", total: { $sum: 1 } } },
        { $project: { _id: 0, pais: "$_id", total: 1 } }
      ],
      as: "usuarios_por_pais"
    }
  },
  {
    $lookup: {
      from: "rooms",
      let: { d_id: "$_id" },
      pipeline: [
        { $match: { $expr: { $eq: ["$dungeon_id", "$$d_id"] } } },
        { $unwind: { path: "$monsters", preserveNullAndEmptyArrays: false } },
        { $group: { 
            _id: null, 
            nivel_mediano_array: { $percentile: { p: [0.5], input: "$monsters.level", method: "approximate" } } 
        } },
        { $project: { _id: 0, nivel_mediano: { $arrayElemAt: ["$nivel_mediano_array", 0] } } }
      ],
      as: "datos_nivel"
    }
  },
  {
    $lookup: {
      from: "rooms",
      let: { d_id: "$_id" },
      pipeline: [
        { $match: { $expr: { $eq: ["$dungeon_id", "$$d_id"] } } },
        { $unwind: { path: "$loot", preserveNullAndEmptyArrays: false } },
        { $group: { _id: null, oro_total: { $sum: "$loot.gold" } } },
        { $project: { _id: 0, oro_total: 1 } }
      ],
      as: "datos_oro"
    }
  },
  {
    $project: {
      _id: 0,
      dungeon_name: 1,
      comentarios_por_tipo: 1,
      usuarios_por_pais: 1,
      nivel_mediano: { $ifNull: [{ $arrayElemAt: ["$datos_nivel.nivel_mediano", 0] }, 0] },
      oro_total: { $ifNull: [{ $arrayElemAt: ["$datos_oro.oro_total", 0] }, 0] }
    }
  }
]

## Query 4

En la colección monsters

```python
[
  {
    $unwind: "$in_rooms"
  },
  {
    $group: {
      _id: "$type",
      dungeons: {
        $addToSet: "$in_rooms.dungeon_name"
      }
    }
  },
  {
    $project: {
      _id: 0,
      monster_type: "$_id",
      dungeons: 1
    }
  }
]

## Query 5


En la colección monsters

```python
[
  {
    $unwind: "$in_rooms"
  },
  {
    $group: {
      _id: {
        dungeon: "$in_rooms.dungeon_name",
        type: "$type"
      },
      cantidad_total: {
        $sum: "$in_rooms.amount"
      }
    }
  },
  {
    $sort: {
      "_id.dungeon": 1,
      cantidad_total: -1
    }
  },
  {
    $group: {
      _id: "$_id.dungeon",
      tipo_mas_comun: {
        $first: "$_id.type"
      }
    }
  },
  {
    $project: {
      _id: 0,
      dungeon_name: "$_id",
      tipo_mas_comun: 1
    }
  }
]

## Query 6

En la colección rooms

```python
[
  {
    $match: {
      monsters: {
        $ne: null
      }
    }
  },
  {
    $addFields: {
      nivel_medio_encuentro: {
        $avg: "$monsters.level"
      }
    }
  },
  {
    $group: {
      _id: null,
      maximo_absoluto: {
        $max: "$nivel_medio_encuentro"
      },
      habitaciones: {
        $push: {
          nivel_medio: "$nivel_medio_encuentro",
          comentarios: "$hints",
          room_id: "$room_id"
        }
      }
    }
  },
  {
    $unwind: "$habitaciones"
  },
  {
    $match: {
      $expr: {
        $eq: [
          "$habitaciones.nivel_medio",
          "$maximo_absoluto"
        ]
      }
    }
  },
  {
    $project: {
      _id: 0,
      comentarios: "$habitaciones.comentarios",
      room_id: "$habitaciones.room_id"
    }
  }
]

## Query 7

En la colección rooms

```python
[
  {
    $match: {
      loot: {
        $ne: null
      },
      monsters: {
        $ne: null
      }
    }
  },
  {
    $addFields: {
      total_oro: {
        $sum: "$loot.gold"
      },
      nivel_medio: {
        $avg: "$monsters.level"
      }
    }
  },
  {
    $addFields: {
      ratio_oro_nivel: {
        $divide: ["$total_oro", "$nivel_medio"]
      }
    }
  },
  {
    $sort: {
      ratio_oro_nivel: 1
    }
  },
  {
    $group: {
      _id: null,
      ratio_array: {
        $push: "$ratio_oro_nivel"
      },
      rooms: {
        $push: {
          _id: "$_id",
          ratio: "$ratio_oro_nivel",
          room_id: "$room_id"
        }
      }
    }
  },
  {
    $addFields: {
      cuartiles: {
        $percentile: {
          input: "$ratio_array",
          p: [0.25, 0.75],
          method: "approximate"
        }
      }
    }
  },
  {
    $addFields: {
      q1: {
        $arrayElemAt: ["$cuartiles", 0]
      },
      q3: {
        $arrayElemAt: ["$cuartiles", 1]
      }
    }
  },
  {
    $addFields: {
      iqr: {
        $subtract: ["$q3", "$q1"]
      }
    }
  },
  {
    $addFields: {
      lower: {
        $subtract: [
          "$q1",
          {
            $multiply: [1.5, "$iqr"]
          }
        ]
      },
      upper: {
        $add: [
          "$q3",
          {
            $multiply: [1.5, "$iqr"]
          }
        ]
      }
    }
  },
  {
    $unwind: "$rooms"
  },
  {
    $match: {
      $expr: {
        $or: [
          {
            $lt: ["$rooms.ratio", "$lower"]
          },
          {
            $gt: ["$rooms.ratio", "$upper"]
          }
        ]
      }
    }
  },
  {
    $project: {
      _id: 0,
      room_id: "$rooms.room_id",
      ratio: "$rooms.ratio"
    }
  }
]

En este caso, hemos considerado solamente solamente las habitaciones que tienen tanto loot como monstruos, para evitar divisiones por cero o ratios sin sentido (en todo caso, esas habitaciones también podrían entrar como outliers). 